# Data Sanity checks

## Imports

In [ ]:
import importlib
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.ticker import FuncFormatter, MultipleLocator, PercentFormatter

# TODO: Why there is so much machinery to import functions? Can we simplify them?
run_mod = importlib.import_module("thesis_project.collectors.run_collectors")
run_mod = importlib.reload(run_mod)

run_collectors_multi = run_mod.run_collectors_multi
infer_lob_type = run_mod.infer_lob_type
iter_lob_parquets = run_mod.iter_lob_parquets
load_lob_dataframe = run_mod.load_lob_dataframe

integrity_mod = importlib.import_module("thesis_project.collectors.integrity")
micro_mod = importlib.import_module("thesis_project.collectors.microstructure")
rel_mod = importlib.import_module("thesis_project.collectors.relationships")

integrity_mod = importlib.reload(integrity_mod)
micro_mod = importlib.reload(micro_mod)
rel_mod = importlib.reload(rel_mod)

collect_timestamp_bounds_series = integrity_mod.collect_timestamp_bounds_series
collect_nan_profile_series = integrity_mod.collect_nan_profile_series
collect_sampling_gaps_series = integrity_mod.collect_sampling_gaps_series
collect_ctd_opening_nan_diagnostics_series = (
    integrity_mod.collect_ctd_opening_nan_diagnostics_series
)
collect_l1_validity_jumps_series = integrity_mod.collect_l1_validity_jumps_series
collect_l1_validity_jump_stats_series = integrity_mod.collect_l1_validity_jump_stats_series
collect_negative_volume_series = integrity_mod.collect_negative_volume_series
collect_price_ordering_series = integrity_mod.collect_price_ordering_series
collect_empty_raw_parquet_files = integrity_mod.collect_empty_raw_parquet_files
collect_missing_dates = integrity_mod.collect_missing_dates

collect_spread_series = micro_mod.collect_spread_series
collect_mid_returns_series = micro_mod.collect_mid_returns_series
collect_depth_imbalance_series = micro_mod.collect_depth_imbalance_series

load_daily_cf = rel_mod.load_daily_cf
collect_basis_series = rel_mod.collect_basis_series

## Paths

In [ ]:
ticker = "FBTP"

root = Path("..")
lob_path = root / "data" / "raw" / ticker
fut_path = lob_path / "futures"
ctd_path = lob_path / "ctd"

year = "2022"
month = "09"
day = "07"
date_str = f"{year}_{month}_{day}"

fut_filename = f"futures_lob_freq_1s_{date_str}.parquet"
ctd_filename = f"ctd_lob_freq_1s_{date_str}.parquet"

# Shared L1 validity settings (same window for futures and CTD).
l1_columns = ["L1-BidPrice", "L1-BidSize", "L1-AskPrice", "L1-AskSize"]
analysis_start_time = "09:00"
analysis_end_time = "17:00"

out_dir = root / "data" / "interim" / ticker
out_dir.mkdir(parents=True, exist_ok=True)

fig_dir = root / "results" / "figures" / "data_integrity"
fig_dir.mkdir(parents=True, exist_ok=True)
tab_dir = root / "results" / "tables" / "data_integrity"
tab_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
norm_cf = True
plot_start_time = "09:00"
plot_end_time = "17:30"

fut_df = pd.read_parquet(fut_path / year / month / fut_filename)
ctd_df = pd.read_parquet(ctd_path / year / month / ctd_filename)
fut_mid = fut_df["MidPrice"]
ctd_mid = ctd_df["MidPrice"]

if norm_cf:
    cf_df = pd.read_csv(lob_path / "daily_cf.csv")
    cf = cf_df[cf_df["Date"] == f"{year}-{month}-{day}"]["CF"].values[0]
    fut_mid = cf * fut_mid

# Filter by time range
start_time = pd.Timestamp(plot_start_time).time()
end_time = pd.Timestamp(plot_end_time).time()
fut_mask = (fut_mid.index.time >= start_time) & (fut_mid.index.time <= end_time)  # type: ignore
ctd_mask = (ctd_mid.index.time >= start_time) & (ctd_mid.index.time <= end_time)  # type: ignore
fut_mid = fut_mid[fut_mask]
ctd_mid = ctd_mid[ctd_mask]

fig, ax1 = plt.subplots(figsize=(12, 6))

if norm_cf:
    # Single y-axis case
    ax1.plot(
        fut_mid.index, fut_mid.values, label="Futures (CF-Normalized)", color="blue", linewidth=1.5
    )
    ax1.plot(ctd_mid.index, ctd_mid.values, label="CTD", color="orange", linewidth=1.5)
    ax1.set_ylabel("Mid Price")
    ax1.legend(loc="best", fontsize=9)
else:
    # Dual y-axis case
    ax1.plot(fut_mid.index, fut_mid.values, label="Futures", color="blue", linewidth=1.5)
    ax1.set_ylabel("Futures Mid Price", color="blue")
    ax1.tick_params(axis="y", labelcolor="blue")

    ax2 = ax1.twinx()
    ax2.plot(ctd_mid.index, ctd_mid.values, label="CTD", color="orange", linewidth=1.5)
    ax2.set_ylabel("CTD Mid Price", color="orange")
    ax2.tick_params(axis="y", labelcolor="orange")
    ax2.grid(False)

ax1.set_xlabel("Time (hh:mm)")

# Build explicit tz-aware bounds
tz = fut_mid.index.tz
start_dt = pd.Timestamp(f"{year}-{month}-{day} {plot_start_time}", tz=tz)
end_dt = pd.Timestamp(f"{year}-{month}-{day} {plot_end_time}", tz=tz)

# Force exact x range and remove margins
left_lim = start_dt - pd.Timedelta(minutes=30)
right_lim = end_dt + pd.Timedelta(minutes=30)
ax1.set_xlim(left_lim, right_lim)
# ax1.margins(x=0)

# Force formatter/locator timezone
ax1.xaxis.set_major_locator(mdates.MinuteLocator(interval=30, tz=tz))
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M", tz=tz))

ax1.tick_params(axis="x", rotation=45)

if norm_cf:
    title = f"{ticker} Futures and CTD Mid Prices on {date_str.replace('_', '/')} (CF-Normalized) from {plot_start_time} to {plot_end_time}"
else:
    title = "Futures vs CTD Mid Prices (Dual Axis)"

ax1.set_title(title)
ax1.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Plot settings and utilities

In [ ]:
# Global plotting defaults
plt.rcParams.update(
    {
        "figure.figsize": (12, 6),
        "figure.dpi": 120,
        "savefig.dpi": 300,
        "font.size": 11,
        "axes.titlesize": 12,
        "axes.labelsize": 11,
        "legend.fontsize": 10,
        "axes.grid": True,
        "grid.alpha": 0.25,
    }
)


def save_figure(fig, stem: str):
    fig.savefig(fig_dir / f"{stem}.png", bbox_inches="tight")


def save_table(df: pd.DataFrame, stem: str):
    df.to_excel(tab_dir / f"{stem}.xlsx", index=False)


def load_output_df(name: str):
    path = out_dir / f"{name}.parquet"
    if path.exists():
        return pd.read_parquet(path)
    raise FileNotFoundError(f"Output file not found: {path}")

# Collectors

## Dataset integrity

In [ ]:
integrity_extractors = {
    # "timestamp_bounds_series": collect_timestamp_bounds_series,
    # "nan_profile_series": collect_nan_profile_series,
    # "sampling_gaps_series": collect_sampling_gaps_series,
    # "ctd_opening_nan_diagnostics_series": collect_ctd_opening_nan_diagnostics_series,
    # "l1_validity_jumps_series": partial(
    #     collect_l1_validity_jumps_series,
    #     l1_columns=l1_columns,
    #     start_time=analysis_start_time,
    #     end_time=analysis_end_time,
    # ),
    # "l1_validity_jump_stats_series": partial(
    #     collect_l1_validity_jump_stats_series,
    #     l1_columns=l1_columns,
    #     start_time=analysis_start_time,
    #     end_time=analysis_end_time,
    # ),
    "negative_volume_series": collect_negative_volume_series,
    "price_ordering_series": collect_price_ordering_series,
}

fut_integrity_results = run_collectors_multi(fut_path, extractors=integrity_extractors)
ctd_integrity_results = run_collectors_multi(ctd_path, extractors=integrity_extractors)

# fut_missing_dates = collect_missing_dates(fut_path)
# ctd_missing_dates = collect_missing_dates(ctd_path)

## Missing LOBs

In [ ]:
raw_root = root / "data" / "raw"
tickers_for_empty_check = ["FBTP", "FBTS"]
lob_dirs = {"futures": "futures", "ctd": "ctd"}

empty_checks_parts = []
for ticker_name in tickers_for_empty_check:
    ticker_root = raw_root / ticker_name
    for lob_label, lob_folder in lob_dirs.items():
        target_path = ticker_root / lob_folder
        if not target_path.exists():
            continue
        part = collect_empty_raw_parquet_files(target_path, lob_type=lob_label)
        if part.empty:
            continue
        part["ticker"] = ticker_name
        empty_checks_parts.append(part)

empty_raw_files_df = (
    pd.concat(empty_checks_parts, ignore_index=True) if empty_checks_parts else pd.DataFrame()
)
display(empty_raw_files_df.head())

if not empty_raw_files_df.empty:
    empty_raw_summary = (
        empty_raw_files_df.groupby(["ticker", "lob_type"], dropna=False)
        .agg(
            total_files=("filename", "count"),
            empty_files=("is_empty", lambda s: int(s.fillna(False).sum())),
            unreadable_files=("read_error", lambda s: int(s.notna().sum())),
        )
        .reset_index()
    )
    display(empty_raw_summary)

## LOBs microstructure

In [ ]:
micro_extractors = {
    "spread_series": collect_spread_series,
    "mid_returns_series": collect_mid_returns_series,
    "depth_imbalance_series": collect_depth_imbalance_series,
}

fut_micro_results = run_collectors_multi(fut_path, extractors=micro_extractors)
ctd_micro_results = run_collectors_multi(ctd_path, extractors=micro_extractors)

cf_path = lob_path / "daily_cf.csv"
basis_parts = []

if cf_path.exists():
    cf_series = load_daily_cf(cf_path)
    ctd_files_by_name = {p.name: p for p in iter_lob_parquets(ctd_path)}
    fut_lob_type = infer_lob_type(fut_path)
    ctd_lob_type = infer_lob_type(ctd_path)

    for fut_file in iter_lob_parquets(fut_path):
        ctd_name = fut_file.name.replace("futures_lob", "ctd_lob")
        ctd_file = ctd_files_by_name.get(ctd_name)
        if ctd_file is None:
            continue

        date_token = fut_file.stem.split("_")[-3:]
        date_key = pd.Timestamp("-".join(date_token)).date()
        if date_key not in cf_series.index:
            continue

        fut_df = load_lob_dataframe(fut_file, fut_lob_type)
        ctd_df = load_lob_dataframe(ctd_file, ctd_lob_type)
        if fut_df is None or ctd_df is None:
            continue

        basis_df = collect_basis_series(fut_df, ctd_df, cf_value=float(cf_series.loc[date_key]))
        if basis_df is None or basis_df.empty:
            continue

        basis_df["fut_filename"] = fut_file.name
        basis_df["ctd_filename"] = ctd_file.name
        basis_parts.append(basis_df)

    basis_series = pd.concat(basis_parts, ignore_index=True) if basis_parts else pd.DataFrame()
    print(f"pair_basis_series: {len(basis_series)} rows")
    display(basis_series.head(5))
else:
    print("daily_cf.csv not found; basis collection skipped.")

# Save data

In [ ]:
for prefix, results in [("fut", fut_integrity_results), ("ctd", ctd_integrity_results)]:
    for key, df in results.items():
        out_path = out_dir / f"{prefix}_{key}.parquet"
        df.to_parquet(out_path, index=False)
        print(f"saved: {out_path}")

if "empty_raw_files_df" in locals() and not empty_raw_files_df.empty:
    empty_raw_out = out_dir / "raw_empty_file_checks.parquet"
    empty_raw_files_df.to_parquet(empty_raw_out, index=False)
    print(f"saved: {empty_raw_out}")

In [ ]:
for family in micro_extractors:
    fut_out = out_dir / f"fut_{family}.parquet"
    ctd_out = out_dir / f"ctd_{family}.parquet"
    fut_micro_results[family].to_parquet(fut_out, index=False)
    ctd_micro_results[family].to_parquet(ctd_out, index=False)
    print(f"saved: {fut_out}")
    print(f"saved: {ctd_out}")

# if "fut_missing_dates" in locals() and "ctd_missing_dates" in locals():
#     fut_missing_out = out_dir / "fut_missing_dates.parquet"
#     ctd_missing_out = out_dir / "ctd_missing_dates.parquet"
#     fut_missing_dates.to_parquet(fut_missing_out, index=False)
#     ctd_missing_dates.to_parquet(ctd_missing_out, index=False)
#     print(f"saved: {fut_missing_out}")
#     print(f"saved: {ctd_missing_out}")

if "basis_series" in locals() and not basis_series.empty:
    basis_out = out_dir / "pair_basis_series.parquet"
    basis_series.to_parquet(basis_out, index=False)
    print(f"saved: {basis_out}")

# Plots and Tables

## Relevant Timestamp Distributions

In [ ]:
lob_type = "fut"
name = f"{lob_type}_timestamp_bounds_series"

ts_df = load_output_df(name)
display(ts_df.head())

columns = ["first_timestamp", "last_timestamp"]
summary = pd.DataFrame(
    {
        "n_unique": ts_df[columns].nunique(),
        "min": ts_df[columns].min(),
        "max": ts_df[columns].max(),
    }
)
columns = [c.replace("_", " ").title() for c in columns]
summary.index = columns
display(summary)

# save_table(summary, f"{lob_type}_timestamp_summary")

In [ ]:
ts_df[ts_df["last_non_nan_timestamp"] == ts_df["last_non_nan_timestamp"].min()].to_excel(
    tab_dir / f"{lob_type}_anomalous_last_timestamp.xlsx", index=False
)

In [ ]:
lob_type = "ctd"
timestamp_col = "last_timestamp"
figure_name = f"{lob_type}_hist_{timestamp_col}"

bin_start_hh_mm = "14:00"
bin_width_seconds = 450
bin_width_title = f"{bin_width_seconds // 60:02d}:{bin_width_seconds % 60:02d}"
major_tick_minutes = 30
minor_tick_minutes = bin_width_seconds / 60
rotate_x_labels = 45
bar_alpha = 0.7
bar_edgecolor = "black"
minor_grid_alpha = 0.15
minor_grid_linestyle = ":"

name = f"{lob_type}_timestamp_bounds_series"
ts_df = load_output_df(name)

ts_parsed = pd.to_datetime(
    ts_df[timestamp_col].astype(str), format="%H:%M:%S", errors="coerce"
).dropna()
x = ts_parsed.dt.hour * 60 + ts_parsed.dt.minute + ts_parsed.dt.second / 60.0

start_h, start_m = map(int, bin_start_hh_mm.split(":"))
start_min = start_h * 60 + start_m
bin_width_min = bin_width_seconds / 60.0
end_min = np.ceil(x.max() / bin_width_min) * bin_width_min
bin_edges = np.arange(start_min, end_min + bin_width_min, bin_width_min)


def min_to_hh_mm(minutes, pos):
    hours = int(minutes // 60)
    mins = int(minutes % 60)
    return f"{hours:02d}:{mins:02d}"


x_formatter = FuncFormatter(min_to_hh_mm)
y_formatter = PercentFormatter(1)

fig, ax = plt.subplots()
ax.hist(
    x,
    bins=bin_edges,  # type: ignore
    weights=np.full(len(x), 1 / len(x)),
    edgecolor=bar_edgecolor,
    alpha=bar_alpha,
)

ax.set_title(
    f"{lob_type.upper()} distribution of {timestamp_col.replace('_', ' ')}\n"
    f"{len(x)} samples, {x.nunique()} unique values\n"
    f"{bin_width_title} bins from {bin_start_hh_mm}:00"
)
ax.set_xlabel("Time (hh:mm)")
ax.set_ylabel("Frequency (%)")

ax.xaxis.set_major_formatter(x_formatter)
ax.xaxis.set_major_locator(MultipleLocator(major_tick_minutes))
ax.xaxis.set_minor_locator(MultipleLocator(minor_tick_minutes))
ax.yaxis.set_major_formatter(y_formatter)

ax.tick_params(axis="x", rotation=rotate_x_labels)
ax.grid(True, which="minor", alpha=minor_grid_alpha, linestyle=minor_grid_linestyle)

plt.show()

# save_figure(fig, figure_name)

## NaN Table


In [ ]:
lob_type = "fut"
sort_by = "pct_nan_total"

name = f"{lob_type}_nan_profile_series"
nan_df = load_output_df(name)

table_cols = [
    "date",
    "filename",
    "pct_nan_total",
    "pct_rows_any_nan",
    "pct_rows_all_nan",
]
nan_table = nan_df[table_cols].sort_values(sort_by, ascending=False).head().reset_index(drop=True)
display(nan_table)

## L1 gaps

In [ ]:
lob_type = "ctd"

events_name = f"{lob_type}_l1_validity_jumps_series"
stats_name = f"{lob_type}_l1_validity_jump_stats_series"

events_df = load_output_df(events_name)
stats_df = load_output_df(stats_name)

print(f"{lob_type} L1 jump events: {len(events_df)}")
display(events_df.head())
display(stats_df.head())

summary_cols = [
    "date",
    "filename",
    "pct_valid_rows_window",
    "n_jumps_gt_1s",
    "mean_jump_seconds",
    "p95_jump_seconds",
    "max_jump_seconds",
    "max_consecutive_invalid_rows",
]

top_jump_days = stats_df[summary_cols].sort_values("max_jump_seconds", ascending=False).head(10)
display(top_jump_days)

# save_table(top_jump_days, f"{lob_type}_l1_validity_jump_top_days")

In [ ]:
lob_type = "ctd"

events_name = f"{lob_type}_l1_validity_jumps_series"
events_df = load_output_df(events_name)

metric_col = "invalid_rows_between_valid"
figure_name = f"{lob_type}_hist_{metric_col}"

# Tunable plotting parameters (same style as Cell 18).
bin_start_value = 0
bin_width = 5
major_tick_step = 10
minor_tick_step = 5
rotate_x_labels = 0
bar_alpha = 0.7
bar_edgecolor = "black"
minor_grid_alpha = 0.15
minor_grid_linestyle = ":"
as_percentage = True

x = events_df[metric_col].dropna()
if x.empty:
    raise ValueError(f"No data available for {metric_col}.")

x_max = int(np.ceil(x.max()))
end_value = x_max + bin_width
bin_edges = np.arange(bin_start_value, end_value + bin_width, bin_width)
if len(bin_edges) < 2:
    bin_edges = np.array([bin_start_value, bin_start_value + bin_width])

weights = np.full(len(x), 1 / len(x)) if as_percentage else None

fig, ax = plt.subplots()
ax.hist(
    x,
    bins=bin_edges,  # type: ignore
    weights=weights,
    edgecolor=bar_edgecolor,
    alpha=bar_alpha,
)

ax.set_title(
    f"{lob_type.upper()} distribution of {metric_col.replace('_', ' ')}\n"
    f"{len(x)} samples, {x.nunique()} unique values\n"
    f"bin width = {bin_width} s"
)
ax.set_xlabel("Consecutive missing rows between two valid L1 samples")
ax.set_xlim(left=bin_start_value - minor_tick_step, right=end_value + minor_tick_step)
ax.set_ylabel("Frequency (%)" if as_percentage else "Frequency (count)")

ax.xaxis.set_major_locator(MultipleLocator(major_tick_step))
ax.xaxis.set_minor_locator(MultipleLocator(minor_tick_step))
if as_percentage:
    ax.yaxis.set_major_formatter(PercentFormatter(1))

ax.tick_params(axis="x", rotation=rotate_x_labels)
ax.grid(True, which="minor", alpha=minor_grid_alpha, linestyle=minor_grid_linestyle)

plt.show()

# save_figure(fig, figure_name)

## Integrity checks: volumes and price ordering

In [ ]:
for lob_type in ["fut", "ctd"]:
    neg_name = f"{lob_type}_negative_volume_series"
    order_name = f"{lob_type}_price_ordering_series"

    neg_df = load_output_df(neg_name)
    order_df = load_output_df(order_name)

    print(f"{lob_type.upper()} negative volume summary")
    display(
        neg_df[
            [
                "date",
                "filename",
                "n_negative_size_cells",
                "n_rows_with_negative_size",
                "has_negative_size",
                "min_size_value",
            ]
        ]
        .sort_values(["n_rows_with_negative_size", "n_negative_size_cells"], ascending=False)
        .head(10)
    )

    print(f"{lob_type.upper()} price ordering summary")
    display(
        order_df[
            [
                "date",
                "filename",
                "n_bid_order_viol_rows",
                "n_ask_order_viol_rows",
                "n_crossed_book_rows",
                "n_any_ordering_violation_rows",
                "has_ordering_violation",
            ]
        ]
        .sort_values("n_any_ordering_violation_rows", ascending=False)
        .head(10)
    )

if "empty_raw_files_df" in locals() and not empty_raw_files_df.empty:
    print("Raw empty-file checks (FBTP + FBTS)")
    display(
        empty_raw_files_df[
            [
                "ticker",
                "lob_type",
                "date",
                "filename",
                "n_rows",
                "is_empty",
                "read_error",
            ]
        ]
        .sort_values(["ticker", "lob_type", "date", "filename"], ascending=True)
        .head(20)
    )

## Sampling gaps

In [ ]:
lob_type = "fut"

name = f"{lob_type}_sampling_gaps_series"
gaps_df = load_output_df(name)
display(gaps_df.head())
print(f"{lob_type} sampling gaps summary:")
print(f"Total number of missing seconds: {gaps_df['missing_points'].sum()}")
print(f"Total number of gaps: {gaps_df['gap_count'].sum()}")
print(f"Greatest gap length: {gaps_df['max_gap_seconds'].max()} seconds")

df = pd.DataFrame(
    {
        "Total missing seconds": gaps_df["missing_points"].sum(),
        "Total number of gaps": gaps_df["gap_count"].sum(),
        "Greatest gap length (s)": gaps_df["max_gap_seconds"].max(),
    },
    index=[lob_type.upper()],
)
display(df)

save_table(df, f"{lob_type}_sampling_gaps_summary")

## Ctd NaN first row

In [ ]:
lob_type = "ctd"

name = f"{lob_type}_opening_nan_diagnostics_series"
nan_df = load_output_df(name)
display(nan_df.head())

## Missing dates

In [ ]:
lob_type = "fut"
name = f"{lob_type}_missing_dates"

missing_dates_df = load_output_df(name)
display(missing_dates_df.head())
print(f"{lob_type} missing dates summary:")
print(f"Total missing dates: {len(missing_dates_df)}")

missing_dates_df.to_csv(out_dir / f"{name}.csv", index=False)